# 🧹 02 · Limpieza y creación de features

En el notebook `01` hicimos de detectives y encontramos 5 problemas. Aquí los **resolvemos**, uno por uno,
mostrando el *antes* y el *después* para que se vea el efecto de cada paso.

**Plan de limpieza (sacado directo de la exploración):**

| # | Problema | Solución |
|---|---|---|
| 1 | Números escondidos como texto (`'21.8%'`) | Convertir a número |
| 2 | Tipo sucio (`indica_sativa`, 15 variantes) | Consolidar a Indica/Sativa/Híbrido |
| 3 | Ratio como texto (`'30% Indica / 70% Sativa'`) | Separar en dos números |
| 4 | Falta de features útiles | Crear columnas nuevas |
| 5 | Columnas basura (constantes/plantilladas) | Eliminar |

> 🧠 **¿Por qué limpiar?** Los análisis y gráficos solo son tan buenos como los datos que reciben.
> "Basura entra, basura sale". Esta es la fase menos vistosa pero **la más importante** de un EDA.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.data_loading import load_raw

pd.set_option("display.max_columns", 50)
df = load_raw()
print("Datos crudos:", df.shape)

Datos crudos: (8910, 38)


## Paso 1 · Convertir texto a número

Las columnas `thc` y `cbd` vienen como `'21.8%'`. Para poder promediar, comparar o graficar, necesitamos
el número `21.8`. La función `pct_to_float` (en `src/data_loading.py`) hace justo eso en 3 pasos:

1. Se asegura de que el valor sea texto.
2. Quita el símbolo `%`.
3. Convierte a número (lo que no se pueda, queda como `NaN`).

Veámosla en acción y comparemos antes/después:

In [2]:
from src.data_loading import pct_to_float

# Creamos las nuevas columnas numéricas (les ponemos el sufijo _pct para que se note que son número)
df["thc_pct"] = pct_to_float(df["thc"])
df["cbd_pct"] = pct_to_float(df["cbd"])

# Antes (texto) vs. después (número)
df[["thc", "thc_pct", "cbd", "cbd_pct"]].head()

,thc,thc_pct,cbd,cbd_pct
0,21.8%,21.8,1.5%,1.5
1,20.9%,20.9,1.0%,1.0
2,24.2%,24.2,1.9%,1.9
3,20.4%,20.4,1.6%,1.6
4,23.0%,23.0,1.5%,1.5


Ahora que son números, pandas ya puede calcular estadísticas. `.describe()` nos da un resumen rápido
(promedio, mínimo, máximo, etc.). Lo entenderemos a fondo en el notebook `03`; por ahora, la prueba de que
la conversión funcionó:

In [3]:
df[["thc_pct", "cbd_pct"]].describe()

,thc_pct,cbd_pct
count,7567.000000,878.000000
mean,20.098980,1.810683
std,2.403629,3.151246
min,-20.000000,0.000000
25%,20.000000,0.425000
50%,20.000000,1.000000
75%,20.000000,1.600000
max,37.800000,28.000000


Las columnas de medidas físicas (`yield_indoor`, `height_indoor`, `flowering_time`...) tienen el mismo
problema pero con otras unidades (`'406.0g/m²'`, `'166.0cm'`, `'64 days'`). Para esas usamos
`extract_number`, que **saca el primer número** de un texto sin importar la unidad:

In [4]:
from src.data_loading import extract_number

df["yield_indoor_num"] = extract_number(df["yield_indoor"])
df["yield_outdoor_num"] = extract_number(df["yield_outdoor"])
df["height_indoor_num"] = extract_number(df["height_indoor"])
df["height_outdoor_num"] = extract_number(df["height_outdoor"])
df["flowering_days"] = extract_number(df["flowering_time"])

df[["flowering_time", "flowering_days", "yield_indoor", "yield_indoor_num"]].head()

,flowering_time,flowering_days,yield_indoor,yield_indoor_num
0,64 days,64.0,406.0g/m²,406.0
1,90 days,90.0,581.0g/m²,581.0
2,60 days,60.0,514.0g/m²,514.0
3,67 days,67.0,606.0g/m²,606.0
4,65 days,65.0,599.0g/m²,599.0


## Paso 2 · Consolidar el tipo de cepa

`indica_sativa` tiene 15 variantes desordenadas (`'Indica Dominant'`, `'50% Indica/50% Sativa'`, e incluso
mezclas raras). Para analizar necesitamos **categorías limpias y pocas**. La función `simplify_type` aplica
una regla clara a cada fila y devuelve solo 3 categorías: **Indica**, **Sativa** o **Híbrido**.

In [5]:
from src.data_loading import simplify_type

df["type_simple"] = simplify_type(df["indica_sativa"])

# Comparamos: valores originales (desordenados) vs. el resultado limpio
print("ANTES (15 variantes):")
print(df["indica_sativa"].value_counts())
print("\nDESPUÉS (3 categorías):")
print(df["type_simple"].value_counts())

ANTES (15 variantes):
indica_sativa
Indica Dominant                                              5254
Sativa Dominant                                              1941
50% Indica/50% Sativa                                        1293
Indica                                                        233
Sativa                                                        141
Indica Dominant , 50% Indica/50% Sativa                        18
Sativa Dominant , 50% Indica/50% Sativa                        16
Indica , Sativa                                                 4
Indica Dominant , Sativa Dominant                               4
50% Indica/50% Sativa , Indica Dominant                         1
50% Indica/50% Sativa , Indica, Indica Dominant                 1
Indica, Sativa Dominant                                         1
50% Indica/50% Sativa , Indica Dominant , Sativa Dominant       1
50% Indica/50% Sativa , Indica, Sativa                          1
Sativa Dominant , Sativa                

## Paso 3 · Separar el ratio Indica/Sativa en números

`type_ratio` trae `'30% Indica / 70% Sativa'`. Ese texto esconde dos números útiles. `parse_type_ratio`
busca el número antes de la palabra "Indica" y el de "Sativa", y los pone en dos columnas nuevas.

In [6]:
from src.data_loading import parse_type_ratio

ratio = parse_type_ratio(df["type_ratio"])
df["indica_pct"] = ratio["indica_pct"]
df["sativa_pct"] = ratio["sativa_pct"]

df[["type_ratio", "indica_pct", "sativa_pct"]].head()

,type_ratio,indica_pct,sativa_pct
0,30% Indica / 70% Sativa,30.0,70.0
1,50% Indica / 50% Sativa,50.0,50.0
2,50% Indica / 50% Sativa,50.0,50.0
3,30% Indica / 70% Sativa,30.0,70.0
4,70% Indica / 30% Sativa,70.0,30.0


## Paso 4 · Crear features nuevas

Una **feature** es una columna nueva que creamos a partir de las que ya hay, para responder preguntas que
los datos crudos no responden directamente. Creamos cuatro:

- `num_effects`, `num_flavors`, `num_medical`: **cuántas** etiquetas tiene cada cepa (contando las comas).
  Sirve para preguntar "¿las cepas con más efectos son más caras?".
- `savings_gbp`: cuánto **ahorras** (precio original − precio actual).

In [7]:
from src.data_loading import count_labels

df["num_effects"] = df["effect"].apply(count_labels)
df["num_flavors"] = df["flavor"].apply(count_labels)
df["num_medical"] = df["medical_strains"].apply(count_labels)
df["savings_gbp"] = (df["original_price_gbp"] - df["current_price_gbp"]).round(2)

df[["effect", "num_effects", "current_price_gbp", "original_price_gbp", "savings_gbp"]].head()

,effect,num_effects,current_price_gbp,original_price_gbp,savings_gbp
0,"Relaxed, Energetic",2,18.54,20.60,2.06
1,"Energetic, Relaxed",2,15.30,17.00,1.70
2,"Euphoric, Focused",2,24.12,26.80,2.68
3,Creative,1,23.68,29.60,5.92
4,"Creative, Euphoric",2,31.40,28.08,-3.32


## Paso 5 · Eliminar las columnas basura

En el notebook `01` identificamos columnas constantes y plantilladas. Las quitamos. La lista está definida
en `src/data_loading.py` como `NOISE_COLS` (así queda documentado el porqué en un solo lugar).

In [8]:
from src.data_loading import NOISE_COLS

print("Columnas a eliminar:", NOISE_COLS)
cols_to_drop = [c for c in NOISE_COLS if c in df.columns]
df = df.drop(columns=cols_to_drop)
print("Forma tras eliminar basura:", df.shape)

Columnas a eliminar: ['sale_item', 'most_popular_seeds', 'experience', 'growth_and_harvest', 'stock_availability']
Forma tras eliminar basura: (8910, 47)


## Bonus · Cómo contaremos las etiquetas múltiples

Las columnas como `effect` guardan varias etiquetas por celda (`'Relaxed, Energetic'`). Para contar cuál es
el efecto más común, hay que **explotar** esas listas: convertir cada etiqueta en su propia fila.
La función `explode_multi` lo hace. Aquí un adelanto de los 10 efectos más frecuentes (los analizaremos
de verdad en el `03`):

In [9]:
from src.data_loading import explode_multi

top_effects = explode_multi(df, "effect").value_counts().head(10)
top_effects

effect
Relaxing     7029
Euphoric      392
Creative      273
Relaxed       269
Energetic     242
Sleepy        208
Focused       201
Uplifting     197
Powerful      159
Happy         136
Name: count, dtype: int64

> 👀 Ojo con `Relaxing`: aparece muchísimo más que el resto. Eso es sospechoso y lo investigaremos en el
> `03` — puede ser un valor por defecto del scraper. Buen ejemplo de por qué **siempre hay que mirar con ojo
> crítico**, incluso después de limpiar.

## Guardar el dataset limpio

Guardamos el resultado en `data/cannabis_clean.csv`. A partir de aquí, los notebooks de análisis cargarán
**este** archivo, ya limpio, en vez del crudo. Así separamos claramente "preparar los datos" de "analizarlos".

In [10]:
from src.data_loading import CLEAN_PATH

df.to_csv(CLEAN_PATH, index=False, encoding="utf-8")
print(f"Guardado en: {CLEAN_PATH}")
print(f"Dataset limpio: {df.shape[0]} filas x {df.shape[1]} columnas")

Guardado en: C:\Users\león\Documents\VacacionesDiciembre2025IA\EDA\cannabis-eda-project\data\cannabis_clean.csv
Dataset limpio: 8910 filas x 47 columnas


### 📝 Resumen del Loop 2

- Convertimos a número: THC, CBD, rendimientos, alturas y días de floración.
- Consolidamos el tipo (15 → 3 categorías) y separamos el ratio en `indica_pct` / `sativa_pct`.
- Creamos features: `num_effects`, `num_flavors`, `num_medical`, `savings_gbp`.
- Eliminamos las columnas basura.
- Guardamos todo en `cannabis_clean.csv`.

> ✅ **Atajo:** toda esta limpieza está empaquetada en la función `clean()` de `src/data_loading.py`.
> En los siguientes notebooks bastará con `df = clean(load_raw())` para tener los datos listos en una línea.

**Siguiente:** `03_eda_narrado` — el análisis de verdad, con la historia químico-medicinal + mercado.